# Resume category classification

Predicts a resume's topic category, not applicant quality or suitability for a job. Duplicate text is removed before the stratified holdout; oversampling is applied only to training data. Old scores and model artifacts are retired because that earlier evaluation leaked duplicated rows across the split. Run all cells to regenerate results and models.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.utils import resample
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


In [ ]:
df = pd.read_csv('clean_resume_data.csv')
df.head()


In [ ]:
df.shape


In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(x='Category', data=df)


In [ ]:
# Remove repeated resume text before reserving a stratified holdout.
df = df.dropna(subset=["Feature", "Category"]).copy()
df["Feature"] = df["Feature"].astype(str).str.strip()
df = df[df["Feature"].ne("")].drop_duplicates(subset=["Feature"])
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df["Category"])
# Oversample only training rows; test data keeps its original class distribution.
max_count = train_df["Category"].value_counts().max()
balanced_df = pd.concat([
    resample(group, replace=len(group) < max_count, n_samples=max_count, random_state=42)
    for _, group in train_df.groupby("Category")
], ignore_index=True)
assert set(balanced_df["Feature"]).isdisjoint(set(test_df["Feature"]))



In [ ]:
balanced_df['Category'].value_counts()


In [ ]:
balanced_df.isnull().sum()


In [ ]:
balanced_df.dropna(inplace=True)


In [ ]:
x_train = balanced_df["Feature"]
y_train = balanced_df["Category"]
x_test = test_df["Feature"]
y_test = test_df["Category"]


In [ ]:
tfidf_vectorizer = TfidfVectorizer()
x_train_tfidf = tfidf_vectorizer.fit_transform(x_train)
x_test_tfidf = tfidf_vectorizer.transform(x_test)


In [ ]:
rf_classifier = RandomForestClassifier(random_state=42, n_jobs=-1)
rf_classifier.fit(x_train_tfidf, y_train)

y_pred = rf_classifier.predict(x_test_tfidf)
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)


In [ ]:
print(classification_report(y_test, y_pred))


In [ ]:
conf_matrix = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(10, 6))
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=rf_classifier.classes_, yticklabels=rf_classifier.classes_)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.title('Confusion Matrix')


In [ ]:
def predict_category(resume_text):
    resume_tfidf = tfidf_vectorizer.transform([resume_text])
    predicted_category = rf_classifier.predict(resume_tfidf)[0]
    return predicted_category


In [ ]:
resume_file = '''hr coordinator summary certified human resources professional extensive employee relations experience full range functions well success simultaneously managing multiple projects highlights employee relations compensation administration personnel records maintenance new hire orientation hiring retention training development compensation payroll staffing recruiting professional boarding interviewing expertise performance management strategies benefits administrator employment law knowledge hris applications proficient employee handbook development new employee orientations human resources audits maintains confidentiality hr policies procedures expertise human resources management excellent interpersonal coaching skills certified professional human resource management accomplishments revamped orientation process new hires implemented company wide led staffing planning committee number years introduced first passive open enrollment process experience current company name city state hr coordinator manage recruitment selection staffing process full time employees manage employee orientation onboarding process full time employees maintain job classification system including job descriptions defining objectives responsibilities salary information benchmarking develop administer manage personnel policies procedures programs city advise managers supervisors employees labor contract employment law policies procedure matters respond inquiries managers supervisors employees investigate complaints provide conflict resolution works departments regarding employee issues recommending appropriate actions involving employee performance behavior productivity etc completes duties related compensation benefits performance management manage city safety workers compensation processes serve chair city benefit safety wellness committees provide direction committees ensures compliance mandated safety training develop wellness safety programs meet strategic goals city develop procedures managing employee leaves absence light duty administer leave absence programs include fmla salary continuation parenting leave military leave etc ensure compliance employment law related regulations conduct research prepare reports recommendations complex issues projects lead special projects related human resources initiatives including software technology implementation process improvements internal training programs boarding process exit interview process etc company name city state independent hr contractor assisted human resources internal employee website mapping project site used employee benefits well company information assisted human resources internal employee website mapping project site used room board sales staff company name city state human resource assistant hr generalist screened applicants internal external positions coordinated prepared interview schedules information packets sent offer letters verified paperwork staffed contractor positions well facilitated orientation contract employees conducted new hire orientations new employees worked levels management employee relations issues conducted exit interviews processed required termination paperwork presented common themes upper management provided training communication employees hr programs benefits processes employment related issues administered leaves absence short term disability worker compensation managed tuition reimbursement program company name city state receptionist administrative assistantmaintained corporate phone list equal employment opportunity information bus pass stamp inventories microsoft excel ordered approved office supply orders coordinated memorial blood centers promote recruit nrg att blood drive company maintained security workplace overseeing security badge process assigned numerous special projects completed projects deadlines education keller graduate school management city state mba human resource management human resource management university north dakota city state b communications communications skills benchmarking benefits conflict resolution direction employee relations performance management personnel policies processes recruitment research safety staffing strategic training programs'''
predicted_category = predict_category(resume_file)
print('Predicted Category: ', predicted_category)


In [ ]:
import pickle
from pathlib import Path
Path('models').mkdir(exist_ok=True)
pickle.dump(rf_classifier, open('models/rf_classifier_categorization.pkl', 'wb'))
pickle.dump(tfidf_vectorizer, open('models/tfidf_vectorizer_categorization.pkl', 'wb'))
